# ⚙️ 05 — Feature Engineering for Grading Error Prediction

**EvaliSense — AI-Powered Handwritten Examination Evaluation**

This notebook demonstrates the 16-dimensional feature vector extracted from each evaluation, used to predict grading errors.

### Feature Groups
| Group | Features | Source |
|-------|----------|--------|
| OCR | `ocr_confidence`, `num_lines`, `avg_line_length`, `text_length` | HTR Pipeline |
| Evaluation | `evaluation_confidence`, `overall_semantic_similarity`, `rubric_coverage` | Evaluator |
| Scoring | `normalized_score`, `max_criterion_score`, `min_criterion_score`, `score_variance` | Score Analysis |
| Keywords | `keyword_match_ratio`, `num_criteria` | Rubric Matching |
| Deviation | `mark_deviation_from_mean_criterion`, `confidence_score_disagreement`, `score_range` | Risk Indicators |

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import matplotlib.pyplot as plt
import time
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Extract Features from a Real Evaluation

In [ ]:
from evaluation import Rubric, AnswerEvaluator
from features import extract_features, FEATURE_NAMES

# Load a rubric and answer
rubric_path = Path('../data/samples/rubric_biology_photosynthesis.json')
answers_path = Path('../data/samples/sample_answers.json')

with open(rubric_path, 'r', encoding='utf-8') as f:
    rubric = Rubric.from_dict(json.load(f))
with open(answers_path, 'r', encoding='utf-8') as f:
    answers_data = json.load(f)

# Initialize evaluator
evaluator = AnswerEvaluator(model_name='all-MiniLM-L6-v2', similarity_threshold=0.55)
print(f'Feature names ({len(FEATURE_NAMES)}):')
for i, name in enumerate(FEATURE_NAMES):
    print(f'  {i+1:2d}. {name}')

In [ ]:
# Extract features for each sample answer that uses this rubric
feature_records = []

for answer_info in answers_data['answers']:
    if 'photosynthesis' not in answer_info['rubric_file']:
        continue
    
    answer_text = answer_info['answer_text']
    quality = answer_info['quality']
    
    # Evaluate
    eval_result = evaluator.evaluate(answer_text, rubric)
    
    # Simulate HTR result (for feature extraction)
    mock_htr = {
        'full_text': answer_text,
        'average_confidence': 0.85 if quality == 'good' else 0.70 if quality == 'partial' else 0.55,
        'lines': [{'text': line, 'confidence': 0.80, 'line_number': i+1}
                  for i, line in enumerate(answer_text.split('\n')) if line.strip()],
    }
    
    # Extract features
    fv = extract_features(mock_htr, eval_result.to_dict())
    feature_dict = fv.to_dict()
    feature_dict['quality'] = quality
    feature_dict['score'] = eval_result.total_score
    feature_dict['answer_id'] = answer_info['id']
    feature_records.append(feature_dict)
    
    print(f'\n{answer_info["id"]} ({quality}): {eval_result.total_score:.1f}/{eval_result.max_score:.0f}')
    print(f'  Feature vector ({len(fv.values)} dims): {np.round(fv.values, 3).tolist()}')

In [ ]:
# Visualize feature values across quality levels
if feature_records:
    # Select key features for visualization
    viz_features = [
        'ocr_confidence', 'overall_semantic_similarity', 'evaluation_confidence',
        'rubric_coverage', 'normalized_score', 'keyword_match_ratio',
        'score_variance', 'confidence_score_disagreement'
    ]
    
    quality_colors = {'good': '#2ECC71', 'partial': '#F39C12', 'poor': '#E74C3C'}
    
    fig, axes = plt.subplots(2, 4, figsize=(20, 8))
    fig.suptitle('Feature Values by Answer Quality', fontsize=16, fontweight='bold')
    
    for ax, feat_name in zip(axes.flat, viz_features):
        for rec in feature_records:
            q = rec['quality']
            val = rec.get(feat_name, 0)
            ax.bar(q, val, color=quality_colors.get(q, '#999'), edgecolor='white', width=0.6)
        ax.set_title(feat_name.replace('_', '\n'), fontsize=9)
        ax.tick_params(axis='x', rotation=30)
    
    plt.tight_layout()
    plt.savefig('../data/results/feature_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

## 2. Analyze Synthetic Training Data Features

In [ ]:
# Load the 300-record synthetic dataset
synthetic_path = Path('../data/samples/synthetic_dataset.jsonl')
syn_records = []

if synthetic_path.exists():
    with open(synthetic_path, 'r', encoding='utf-8') as f:
        for line in f:
            syn_records.append(json.loads(line.strip()))
    
    print(f'Loaded {len(syn_records)} synthetic records')
    errors = [r for r in syn_records if r['is_grading_error']]
    print(f'Grading errors: {len(errors)} ({len(errors)/len(syn_records):.1%})')
    
    # Feature correlation with grading error
    feat_keys = list(syn_records[0]['features'].keys())
    
    print(f'\n{"Feature":<40} {"Non-Error Mean":>14} {"Error Mean":>12} {"Δ":>8}')
    print('─' * 80)
    
    deltas = {}
    for k in feat_keys:
        non_err = np.mean([r['features'][k] for r in syn_records if not r['is_grading_error']])
        err = np.mean([r['features'][k] for r in syn_records if r['is_grading_error']])
        delta = abs(err - non_err)
        deltas[k] = delta
        indicator = '⬆' if err > non_err else '⬇'
        print(f'{k:<40} {non_err:>14.4f} {err:>12.4f} {indicator}{delta:>6.4f}')
else:
    print('Synthetic dataset not found.')

In [ ]:
# Feature importance ranking (by separation between classes)
if syn_records and deltas:
    sorted_features = sorted(deltas.items(), key=lambda x: x[1], reverse=True)
    
    fig, ax = plt.subplots(figsize=(14, 7))
    names = [f[0] for f in sorted_features]
    values = [f[1] for f in sorted_features]
    colors = ['#E74C3C' if v > 0.1 else '#F39C12' if v > 0.05 else '#95A5A6' for v in values]
    
    ax.barh(range(len(names)), values, color=colors, edgecolor='white')
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=9)
    ax.set_xlabel('Mean Absolute Difference (Error vs Non-Error)')
    ax.set_title('Feature Importance: Class Separation', fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    
    # Add value labels
    for i, v in enumerate(values):
        ax.text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.savefig('../data/results/feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Correlation matrix
if syn_records:
    feat_keys = list(syn_records[0]['features'].keys())
    data_matrix = np.array([[r['features'][k] for k in feat_keys] for r in syn_records])
    
    corr = np.corrcoef(data_matrix.T)
    
    fig, ax = plt.subplots(figsize=(14, 12))
    im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xticks(range(len(feat_keys)))
    ax.set_yticks(range(len(feat_keys)))
    short_names = [k.replace('_', '\n')[:20] for k in feat_keys]
    ax.set_xticklabels(short_names, fontsize=7, rotation=45, ha='right')
    ax.set_yticklabels(short_names, fontsize=7)
    ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
    plt.colorbar(im, label='Pearson Correlation')
    
    plt.tight_layout()
    plt.savefig('../data/results/feature_correlation.png', dpi=150, bbox_inches='tight')
    plt.show()

## Summary

The 16-dimensional feature vector captures signals from:
- **OCR quality** — Low recognition confidence → potential misread → wrong score
- **Semantic alignment** — How well the answer matches rubric criteria
- **Score patterns** — High variance or extreme scores flag uncertain grading
- **Keyword coverage** — Missing domain terms indicate potential under-scoring

**Next:** Proceed to `06_model_training.ipynb` for ML model comparison.